In [1]:
# ---------------------------------------------------------------------------
# Common setup and model (enlarged-seeding model of Sections 2-3).
# Fibers are thin cylinders of radius r; midpoints are seeded on the enlarged
# region S = [-lmax/2, 1+lmax/2]^3 and contacts are counted inside the unit
# cube C via the contact-point gate. Grayscale seaborn figures, 600 dpi,
# PNG + PDF, no titles/captions. No paths are printed (kept anonymous).
# ---------------------------------------------------------------------------
import os, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 10, "axes.edgecolor": "0.3", "savefig.dpi": 600})
GRAY, DARK = "0.45", "0.15"

_FIG = os.path.join("..", "results", "figures")
_TAB = os.path.join("..", "results", "tables")
_DAT = os.path.join("..", "data")
for _d in (_FIG, _TAB, _DAT):
    os.makedirs(_d, exist_ok=True)

def savefig(fig, name):
    """Save a figure as PNG and PDF (600 dpi, no title); do not print paths."""
    fig.savefig(os.path.join(_FIG, name + ".png"), bbox_inches="tight")
    fig.savefig(os.path.join(_FIG, name + ".pdf"), bbox_inches="tight")
    plt.close(fig)

def save_table(df, name, float_format="%.4f"):
    df.to_latex(os.path.join(_TAB, name + ".tex"), index=False, float_format=float_format)

def save_csv(df, name):
    df.to_csv(os.path.join(_DAT, name + ".csv"), index=False)

# ---- model constants (two-length mixture) --------------------------------
ELL = np.array([0.2, 0.4]); PMIX = np.array([0.5, 0.5])
LMAX = float(ELL.max()); LBAR = float((PMIX * ELL).sum())
CVOL, SVOL = 1.0, (1.0 + LMAX) ** 3        # |C| = 1, |S| = 1.4^3

def raw_moment(k):
    return float((PMIX * ELL ** k).sum())

def closed_forms(r):
    lam1 = raw_moment(1) * CVOL / SVOL
    lam2 = (raw_moment(2) - 0.5 * raw_moment(3)
            + raw_moment(4) / (3 * np.pi) - raw_moment(5) / (40 * np.pi)) / SVOL
    var_lc = lam2 - lam1 ** 2
    q_lead = np.pi * r * LBAR ** 2 * CVOL / SVOL ** 2
    q_fin = q_lead * (1 + 8 * r / LBAR)                 # Appendix D end term
    zeta1 = (np.pi * r * LBAR / SVOL) ** 2 * var_lc     # leading order
    zeta1_fin = zeta1 * (1 + 8 * r / LBAR) ** 2         # finite-r enhancement
    return dict(lam1=lam1, lam2=lam2, var_lc=var_lc,
                q_lead=q_lead, q_fin=q_fin, zeta1=zeta1, zeta1_fin=zeta1_fin)

def nstar(r):
    return CVOL / (2 * np.pi * r * closed_forms(r)["var_lc"])

def var_components(n, r, finite=True):
    cf = closed_forms(r); q = cf["q_fin"]
    z1 = cf["zeta1_fin"] if finite else cf["zeta1"]
    s2 = q * (1 - q) - 2 * z1
    vp = n * (n - 1) * (n - 2) * z1
    vr = (n * (n - 1) / 2) * s2
    return vp, vr, vp + vr

def iso_dirs(n, rng):
    v = rng.normal(size=(n, 3)); v /= np.linalg.norm(v, axis=1, keepdims=True)
    return v

def oblate_dirs(n, rng, c=0.5):
    v = iso_dirs(n, rng); v[:, 2] *= c
    v /= np.linalg.norm(v, axis=1, keepdims=True)
    return v

def seg_seg(P1, P2, Q1, Q2, want_mid=False):
    """Minimum distance between segments [P1,P2] and [Q1,Q2] (clamped
    Lumelsky scheme), vectorized over rows; optionally the closest-point
    midpoint."""
    d1, d2, r = P2 - P1, Q2 - Q1, P1 - Q1
    a = (d1 * d1).sum(1); e = (d2 * d2).sum(1); f = (d2 * r).sum(1)
    b = (d1 * d2).sum(1); c = (d1 * r).sum(1)
    den = a * e - b * b
    s = np.where(den > 1e-12, np.clip((b * f - c * e) / den, 0, 1), 0.0)
    t = np.clip((b * s + f) / e, 0, 1)
    s = np.clip((b * t - c) / a, 0, 1)
    cpA, cpB = P1 + s[:, None] * d1, Q1 + t[:, None] * d2
    dist = np.linalg.norm(cpA - cpB, axis=1)
    if want_mid:
        return dist, 0.5 * (cpA + cpB)
    return dist

def sample_fibers(n, rng, dirfun=None):
    """Enlarged seeding: midpoints uniform on S; returns segment endpoints."""
    L = rng.choice(ELL, size=n, p=PMIX)
    X = rng.uniform(-LMAX / 2, 1 + LMAX / 2, size=(n, 3))
    v = iso_dirs(n, rng) if dirfun is None else dirfun(n, rng)
    return X - 0.5 * L[:, None] * v, X + 0.5 * L[:, None] * v

def count_once(n, twor, rng, dirfun=None):
    """One realization: enlarged seeding + contact-point-in-C gate."""
    P1, P2 = sample_fibers(n, rng, dirfun)
    i, j = np.triu_indices(n, 1)
    dist, mid = seg_seg(P1[i], P2[i], P1[j], P2[j], want_mid=True)
    inC = np.all((mid >= 0) & (mid <= 1), axis=1)
    return int(np.sum((dist <= twor) & inC))

# === 07_anisotropy_and_timing.ipynb ========================================
# (a) Robustness to the directional distribution: isotropic vs oblate
#     (in-plane preferred, c = 0.5) under enlarged seeding, with the
#     end-term-corrected mean prediction. (b) Wall-clock cost per realization,
#     confirming the Theta(n^2) pair enumeration.
import time
rng = np.random.default_rng(555); r, twor = 0.01, 0.02; cf = closed_forms(r)

def mean_sine(dirfun, m=1_000_000):
    a, b = dirfun(m, rng), dirfun(m, rng)
    return float(np.linalg.norm(np.cross(a, b), axis=1).mean())

Es = {"isotropic": mean_sine(iso_dirs),
      "oblate_c0.5": mean_sine(lambda n, g: oblate_dirs(n, g, 0.5))}
frac_lead = 1.0 / (1 + 8 * r / LBAR)
# publication run: reps = [2500, 1500]
arows = []
for law, dfun in [("isotropic", iso_dirs),
                  ("oblate_c0.5", lambda n, g: oblate_dirs(n, g, 0.5))]:
    for n, reps in [(400, 800), (800, 400)]:
        g = np.random.default_rng(5000 + n + (0 if law == "isotropic" else 11))
        c = np.array([count_once(n, twor, g, dfun) for _ in range(reps)], float)
        EN = (n * (n - 1) / 2) * cf["q_fin"]
        pred = EN * (frac_lead * (Es[law] / (np.pi / 4)) + (1 - frac_lead))
        arows.append(dict(law=law, n=n, Esin=Es[law], mean=c.mean(),
                          mean_pred=pred, skew=float(stats.skew(c))))
atab = pd.DataFrame(arows)
save_table(atab, "table_anisotropy", float_format="%.4f"); save_csv(atab, "anisotropy")
print(atab.to_string(index=False))

trows = []; prev = None
for n in [100, 200, 400, 800, 1600]:
    reps = max(5, int(2e7 / (n * n)))
    count_once(n, twor, rng)                          # warm-up
    t = time.time()
    for _ in range(reps):
        count_once(n, twor, rng)
    ms = (time.time() - t) / reps * 1000
    trows.append(dict(n=n, ms_per_realization=ms,
                      ratio=(ms / prev if prev else np.nan))); prev = ms
ttab = pd.DataFrame(trows)
save_table(ttab, "table_timing", float_format="%.2f"); save_csv(ttab, "timing")
print(); print(ttab.to_string(index=False))

        law   n     Esin      mean  mean_pred     skew
  isotropic 400 0.785578  38.35125  37.963662 0.254078
  isotropic 800 0.785578 155.65000 152.044942 0.000153
oblate_c0.5 400 0.756364  37.15125  36.849037 0.149228
oblate_c0.5 800 0.756364 148.84000 147.580854 0.017951



   n  ms_per_realization    ratio
 100            2.006549      NaN
 200            8.679254 4.325463
 400           37.510447 4.321851
 800          142.150187 3.789616
1600          655.064923 4.608259
